# M04A Scientific Screening Analysis (`M04A_20261126_145808_DESKTOP-CQB09OU`)

**Scope:** Clean-data Protocol A exploratory screening (`train=12,216` rows / `2,036` gridline groups; `validation=3,060` rows / `510` gridline groups; `seed=42`; `scientific_v1` prespecified configurations).
**Governance:** Historical `test` (`3,822` rows) remains strictly sealed and excluded. Exploratory validation results are NOT final test results.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()
RUN_ID = "M04A_20261126_145808_DESKTOP-CQB09OU"
RUN_DIR = ROOT / "reports" / "runs" / RUN_ID
FIG_DIR = RUN_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 140, "axes.grid": True, "grid.alpha": 0.25})
print("Run directory:", RUN_DIR)

## 1–3. Dataset, Split Verification & Model Status Summary

In [ ]:
split_verif = json.loads((RUN_DIR / "M04A_SPLIT_VERIFICATION.json").read_text(encoding="utf-8"))
data_audit = json.loads((RUN_DIR / "M04A_DATA_ACCESS_AUDIT.json").read_text(encoding="utf-8"))
status_df = pd.read_csv(RUN_DIR / "M04A_MODEL_STATUS.csv")
raw_metrics = pd.read_csv(RUN_DIR / "M04A_VALIDATION_METRICS_RAW.csv")
clip_metrics = pd.read_csv(RUN_DIR / "M04A_VALIDATION_METRICS_NONNEGATIVE.csv")
ref_metrics = pd.read_csv(RUN_DIR / "M04A_REFERENCE_VALIDATION_METRICS.csv")
res_df = pd.read_csv(RUN_DIR / "M04A_RESOURCE_USAGE.csv")
sub_df = pd.read_csv(RUN_DIR / "M04A_SUBGROUP_METRICS.csv")
pred_df = pd.read_csv(RUN_DIR / "M04A_VALIDATION_PREDICTIONS.csv")

print("Raw CSV shape:", split_verif["raw_csv_shape"], "| Test rows exposed:", data_audit["test_rows_exposed_to_worker"])
print("M04A Status counts:\n", status_df["m04a_status"].value_counts())

## 4. Scientific Validation Leaderboard (Raw & Nonnegative-Clipped + Historical Reference Provenance)

In [ ]:
display_cols = ["rank_raw_mae", "model_id", "mae_um", "rmse_um", "lins_ccc", "r2", "bias_um", "within_20um_rate", "n_negative_raw_predictions"]
print("=== NEWLY TRAINED / EVALUATED MODELS (RAW PREDICTIONS, PROTOCOL A VALIDATION) ===")
print(raw_metrics[display_cols].to_string(index=False))
print("\n=== NONNEGATIVE-CLIPPED VARIANT ===")
print(clip_metrics[["rank_clipped_mae", "model_id", "mae_um", "rmse_um", "lins_ccc", "r2"]].to_string(index=False))
print("\n=== HISTORICAL REFERENCE OUTPUTS (SEPARATE PROVENANCE — NOT FAIR RETRAINING) ===")
print(ref_metrics[["model_id", "mae_um", "rmse_um", "lins_ccc", "r2", "provenance"]].to_string(index=False))

## 5. MAE and RMSE Comparison Across Candidates

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
plot_df = raw_metrics.sort_values("mae_um", ascending=True)
y_pos = np.arange(len(plot_df))
ax.barh(y_pos - 0.18, plot_df["mae_um"], height=0.35, label="Raw MAE (um)", color="#1f77b4")
ax.barh(y_pos + 0.18, plot_df["rmse_um"], height=0.35, label="Raw RMSE (um)", color="#ff7f0e")
ax.set_yticks(y_pos)
ax.set_yticklabels(plot_df["model_id"])
ax.invert_yaxis()
ax.set_xlabel("Error (um) — Protocol A Validation (n=3,060)")
ax.set_title(f"M04A Validation MAE & RMSE Comparison ({RUN_ID})")
ax.legend()
plt.tight_layout()
fig.savefig(FIG_DIR / "fig01_mae_rmse_comparison.png")
plt.show()

## 6–8. Actual vs Predicted, Error Distributions & Bland-Altman (Top-4 Models)

In [ ]:
top4 = list(raw_metrics["model_id"].head(4))
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
for ax, mid in zip(axes.flat, top4):
    sub = pred_df[pred_df["model_id"] == mid]
    ax.scatter(sub["y_true_um"], sub["y_pred_raw_um"], s=6, alpha=0.35)
    lim = max(sub["y_true_um"].max(), sub["y_pred_raw_um"].max())
    ax.plot([0, lim], [0, lim], "r--", lw=1)
    mae_v = raw_metrics.loc[raw_metrics["model_id"] == mid, "mae_um"].values[0]
    ax.set_title(f"{mid} (MAE={mae_v:.2f} um, n={len(sub)})")
    ax.set_xlabel("Actual MANwidth (um)")
    ax.set_ylabel("Predicted Raw (um)")
plt.suptitle(f"Actual vs Predicted — Top-4 Models ({RUN_ID})")
plt.tight_layout()
fig.savefig(FIG_DIR / "fig02_actual_vs_predicted_top4.png")
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, mid in zip(axes.flat, top4):
    sub = pred_df[pred_df["model_id"] == mid]
    err = sub["raw_signed_error_um"]
    ax.hist(err, bins=50, color="#2ca02c", alpha=0.75)
    ax.axvline(0.0, color="k", linestyle="--", lw=1)
    ax.set_title(f"{mid} Signed Error (Pred - True)")
    ax.set_xlabel("Signed Error (um)")
    ax.set_ylabel("Count")
plt.suptitle(f"Error Distributions — Top-4 Models ({RUN_ID})")
plt.tight_layout()
fig.savefig(FIG_DIR / "fig03_error_distributions_top4.png")
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, mid in zip(axes.flat, top4):
    sub = pred_df[pred_df["model_id"] == mid]
    mean_v = 0.5 * (sub["y_true_um"] + sub["y_pred_raw_um"])
    diff_v = sub["y_pred_raw_um"] - sub["y_true_um"]
    md_v = diff_v.mean()
    sd_v = diff_v.std(ddof=1)
    ax.scatter(mean_v, diff_v, s=6, alpha=0.3)
    ax.axhline(md_v, color="r", lw=1.2, label=f"Mean={md_v:.2f}")
    ax.axhline(md_v + 1.96 * sd_v, color="b", ls="--", lw=1, label=f"+1.96 SD={md_v+1.96*sd_v:.1f}")
    ax.axhline(md_v - 1.96 * sd_v, color="b", ls="--", lw=1, label=f"-1.96 SD={md_v-1.96*sd_v:.1f}")
    ax.set_title(f"{mid} Descriptive Bland-Altman (Row-Level)")
    ax.set_xlabel("Mean of True & Pred (um)")
    ax.set_ylabel("Pred - True (um)")
    ax.legend(fontsize=7)
plt.suptitle(f"Descriptive Row-Level Bland-Altman Plots ({RUN_ID})")
plt.tight_layout()
fig.savefig(FIG_DIR / "fig04_bland_altman_top4.png")
plt.show()

## 9–11. Analytical Tolerances, Zero-Width Regime, Per-Series & Per-Stage Breakdown

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
x = np.arange(len(raw_metrics))
w = 0.2
for idx, col in enumerate(["within_5um_rate", "within_10um_rate", "within_20um_rate", "within_50um_rate"]):
    ax.bar(x + (idx - 1.5) * w, raw_metrics[col] * 100.0, width=w, label=col.replace("_rate", ""))
ax.set_xticks(x)
ax.set_xticklabels(raw_metrics["model_id"], rotation=45, ha="right")
ax.set_ylabel("Hit Rate (%)")
ax.set_title(f"Analytical Tolerance Hit Rates ({RUN_ID}, n=3,060)")
ax.legend()
plt.tight_layout()
fig.savefig(FIG_DIR / "fig05_tolerance_hit_rates.png")
plt.show()

zw = sub_df[sub_df["subgroup_value"] == "zero_width_0"].merge(raw_metrics[["model_id", "rank_raw_mae"]], on="model_id").sort_values("rank_raw_mae")
print("=== ZERO-WIDTH REGIME (MANwidth == 0.0 um) ===")
print(zw[["model_id", "n_obs", "raw_mae_um", "clipped_mae_um", "raw_bias_um"]].to_string(index=False))

ser = sub_df[sub_df["subgroup_type"] == "Series"].pivot(index="model_id", columns="subgroup_value", values="raw_mae_um")
print("\n=== PER-SERIES RAW MAE (um) ===")
print(ser.loc[raw_metrics["model_id"]].round(2).to_string())

## 12–15. Failures, Computational Cost & Accuracy-Efficiency Pareto

In [ ]:
print("=== FAILED / NON-M04A MODELS ===")
print(status_df[status_df["m04a_status"] != "PASS"][["model_id", "name", "m04a_status", "failure_or_exclusion_reason"]].to_string(index=False))

pareto = raw_metrics.merge(res_df[["model_id", "total_elapsed_sec", "device"]], on="model_id")
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(pareto["total_elapsed_sec"], pareto["mae_um"], s=60, alpha=0.8)
for _, row in pareto.iterrows():
    ax.annotate(row["model_id"], (row["total_elapsed_sec"], row["mae_um"]), fontsize=8, xytext=(4, 2), textcoords="offset points")
ax.set_xscale("log")
ax.set_xlabel("Total Elapsed Time (s, log scale) — Single-Run Screening")
ax.set_ylabel("Raw Validation MAE (um)")
ax.set_title(f"Accuracy vs Runtime Pareto ({RUN_ID})")
plt.tight_layout()
fig.savefig(FIG_DIR / "fig06_accuracy_efficiency_pareto.png")
plt.show()